In [35]:
import pandas as pd
df = pd.read_csv("/content/cleaned_76hd.csv")
df.shape

(282, 39)

In [36]:
X = df.drop(columns=["V58"])
Y = df["V58"]

In [37]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=7,stratify=Y)
X_train.shape, X_test.shape, Y_train.shape, Y_test.shape

((197, 38), (85, 38), (197,), (85,))

In [38]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

pg = {"C":[0.1,0.5,1.0],"gamma":[0.5,1,5]}

# Create an SVC estimator with the desired kernel and random_state
svc_estimator = SVC(kernel="rbf", random_state=7)

# Pass the SVC estimator to GridSearchCV
gscv = GridSearchCV(estimator=svc_estimator, param_grid=pg, cv=2, scoring="average_precision", verbose=2)
gscv.fit(X_train,Y_train)

Fitting 2 folds for each of 9 candidates, totalling 18 fits
[CV] END ...................................C=0.1, gamma=0.5; total time=   0.0s
[CV] END ...................................C=0.1, gamma=0.5; total time=   0.0s
[CV] END .....................................C=0.1, gamma=1; total time=   0.0s
[CV] END .....................................C=0.1, gamma=1; total time=   0.0s
[CV] END .....................................C=0.1, gamma=5; total time=   0.0s
[CV] END .....................................C=0.1, gamma=5; total time=   0.0s
[CV] END ...................................C=0.5, gamma=0.5; total time=   0.0s
[CV] END ...................................C=0.5, gamma=0.5; total time=   0.0s
[CV] END .....................................C=0.5, gamma=1; total time=   0.0s
[CV] END .....................................C=0.5, gamma=1; total time=   0.0s
[CV] END .....................................C=0.5, gamma=5; total time=   0.0s
[CV] END .....................................C=0

GridSearchCV(cv=2, estimator=SVC(random_state=7),
             param_grid={'C': [0.1, 0.5, 1.0], 'gamma': [0.5, 1, 5]},
             scoring='average_precision', verbose=2)

In [39]:
gscv.best_estimator_

SVC(gamma=0.5, random_state=7)

In [40]:
from sklearn.metrics import classification_report


svm = SVC(C=1,gamma=0.5,kernel="rbf",random_state=7)
svm.fit(X_train,Y_train)
Y_pred = svm.predict(X_test)

print(classification_report(Y_test,Y_pred))

              precision    recall  f1-score   support

           0       0.80      0.80      0.80        46
           1       0.77      0.77      0.77        39

    accuracy                           0.79        85
   macro avg       0.79      0.79      0.79        85
weighted avg       0.79      0.79      0.79        85



In [41]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV # Import GridSearchCV

pg = {"max_features":[5,10,20],"n_estimators":[20,100,200]} # Corrected typo and n_estimators values

# Create a RandomForestClassifier estimator
rf_base_estimator = RandomForestClassifier(random_state = 7, oob_score = True, max_samples=0.8)

# Pass the RandomForestClassifier estimator to GridSearchCV
gscv = GridSearchCV(estimator=rf_base_estimator, param_grid=pg, cv=2, scoring="accuracy", verbose=2)
gscv.fit(X_train,Y_train)

# You can then access the best estimator and make predictions, similar to the previous cell:
best_rf_model = gscv.best_estimator_
Y_pred = best_rf_model.predict(X_test)
print("Best parameters found:", gscv.best_params_)
print("Best cross-validation score:", gscv.best_score_)

Fitting 2 folds for each of 9 candidates, totalling 18 fits
[CV] END ....................max_features=5, n_estimators=20; total time=   0.1s
[CV] END ....................max_features=5, n_estimators=20; total time=   0.0s
[CV] END ...................max_features=5, n_estimators=100; total time=   0.2s
[CV] END ...................max_features=5, n_estimators=100; total time=   0.2s
[CV] END ...................max_features=5, n_estimators=200; total time=   0.5s
[CV] END ...................max_features=5, n_estimators=200; total time=   0.7s
[CV] END ...................max_features=10, n_estimators=20; total time=   0.1s
[CV] END ...................max_features=10, n_estimators=20; total time=   0.1s
[CV] END ..................max_features=10, n_estimators=100; total time=   0.3s
[CV] END ..................max_features=10, n_estimators=100; total time=   0.4s
[CV] END ..................max_features=10, n_estimators=200; total time=   0.7s
[CV] END ..................max_features=10, n_est

In [50]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Define the parameter grid for GridSearchCV
pg = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, 15],
    "learning_rate": [ 0.5, 0.3]
}

# Create an XGBClassifier estimator with fixed parameters not being tuned
xgb_base_estimator = XGBClassifier(subsample=0.8, colsample_bytree=0.8, random_state=7, use_label_encoder=False, eval_metric='logloss')

# Pass the XGBClassifier estimator and the parameter grid to GridSearchCV
gscv_xgb = GridSearchCV(estimator=xgb_base_estimator, param_grid=pg, cv=2, scoring="accuracy", verbose=2)
gscv_xgb.fit(X_train, Y_train)

# Get the best model and make predictions
best_xgb_model = gscv_xgb.best_estimator_
Y_pred = best_xgb_model.predict(X_test)

# Print evaluation metrics
print("Best parameters found:", gscv_xgb.best_params_)

print("Classification Report:\n", classification_report(Y_test, Y_pred))

Fitting 2 folds for each of 18 candidates, totalling 36 fits
[CV] END ....learning_rate=0.5, max_depth=5, n_estimators=50; total time=   0.1s
[CV] END ....learning_rate=0.5, max_depth=5, n_estimators=50; total time=   0.1s


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:51:49] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:51:49] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:51:49] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END ...learning_rate=0.5, max_depth=5, n_estimators=100; total time=   0.1s
[CV] END ...learning_rate=0.5, max_depth=5, n_estimators=100; total time=   0.1s


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:51:49] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:51:49] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END ...learning_rate=0.5, max_depth=5, n_estimators=200; total time=   0.6s


...

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:51:52] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:51:52] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:51:52] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:51:52] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END ..learning_rate=0.3, max_depth=10, n_estimators=200; total time=   0.1s
[CV] END ..learning_rate=0.3, max_depth=10, n_estimators=200; total time=   0.1s
[CV] END ...learning_rate=0.3, max_depth=15, n_estimators=50; total time=   0.0s
[CV] END ...learning_rate=0.3, max_depth=15, n_estimators=50; total time=   0.0s
[CV] END ..learning_rate=0.3, max_depth=15, n_estimators=100; total time=   0.0s


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:51:52] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:51:52] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:51:52] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:51:52] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END ..learning_rate=0.3, max_depth=15, n_estimators=100; total time=   0.0s
[CV] END ..learning_rate=0.3, max_depth=15, n_estimators=200; total time=   0.1s
[CV] END ..learning_rate=0.3, max_depth=15, n_estimators=200; total time=   0.1s
Best parameters found: {'learning_rate': 0.5, 'max_depth': 5, 'n_estimators': 100}
Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.83      0.81        46
           1       0.78      0.74      0.76        39

    accuracy                           0.79        85
   macro avg       0.79      0.78      0.79        85
weighted avg       0.79      0.79      0.79        85



/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:51:53] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [55]:
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Define the parameter grid for GridSearchCV
pg = {
    "n_estimators": [1000, 2000, 5000],
    "max_depth": [5, 10, 15, 30],
    "learning_rate": [ 0.1, 0.25, 0.5, 0.75]
}

# Create an XGBClassifier estimator with fixed parameters not being tuned
lgbm = lgb.LGBMClassifier(subsample=0.8, colsample_bytree=0.8, random_state=7, use_label_encoder=False, eval_metric='logloss')

# Pass the XGBClassifier estimator and the parameter grid to GridSearchCV
gscv_lgbm = GridSearchCV(estimator=xgb_base_estimator, param_grid=pg, cv=2, scoring="accuracy", verbose=2)
gscv_lgbm.fit(X_train, Y_train)

# Get the best model and make predictions
lgbm.fit(X_train,Y_train)
Y_pred = lgbm.predict(X_test)

# Print evaluation metrics
print("Best parameters found:", gscv_lgbm.best_params_)
print(gscv_lgbm.best_score_)
print("Classification Report:\n", classification_report(Y_test, Y_pred))

Fitting 2 folds for each of 48 candidates, totalling 96 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:01:19] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END ..learning_rate=0.1, max_depth=5, n_estimators=1000; total time=   1.1s


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:01:20] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END ..learning_rate=0.1, max_depth=5, n_estimators=1000; total time=   0.7s


...

[CV] END learning_rate=0.75, max_depth=30, n_estimators=5000; total time=   0.9s


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:02:17] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END learning_rate=0.75, max_depth=30, n_estimators=5000; total time=   0.9s


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:02:18] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] Unknown parameter: use_label_encoder
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] Unknown parameter: use_label_encoder
[LightGBM] [Info] Number of positive: 89, number of negative: 108
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000104 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 572
[LightGBM] [Info] Number of data points in the train set: 197, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.451777 -> initscore=-0.193495
[LightGBM] [Info] Start training from score -0.193495
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Ligh